# SignalCheck · TF-IDF baseline

A reproducible text classifier. **Sample mode** runs immediately on fictional data and checks the pipeline. Switch to `processed` after preparing WELFake to train on real data. Historical metrics in the root `results/` folder are separate from this run.

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open this notebook from the project root or notebooks/ directory")
sys.path.insert(0, str(ROOT))
from src.pipeline import load_data, split_data, train_baseline, evaluate, save_results

## 1 · Choose data

The bundled sample is synthetic and must not be used to claim real-world accuracy. For the full run, place the prepared CSV at `data/processed/final_dataset.csv`.

In [4]:
DATA_MODE = "sample"  # change to "processed" for the full dataset
frame = load_data(DATA_MODE)
print(f"Rows: {len(frame):,}; labels: {frame.label.value_counts().to_dict()}")
display(frame[["combined", "label"]].head(3))

Rows: 32; labels: {1: 16, 0: 16}
                                                                                         combined  label
Demo City opens a library The fictional city council opened a new library in the sample district.      1
    Demo City repairs a bridge The fictional city council completed repairs to the sample bridge.      1
   Demo City adds bus stops The fictional transit office added three bus stops to a sample route.      1


## 2 · Split and fit

Text is split before the vectorizer is fitted, preventing vocabulary leakage.

In [6]:
train, valid, test = split_data(frame, sample=DATA_MODE == "sample")
print("Train / validation / test:", len(train), len(valid), len(test))
model = train_baseline(train)

Train / validation / test: 19 6 7


## 3 · Evaluate

Validation helps inspect the model. The held-out test split is evaluated once here.

In [8]:
validation_pred = model.predict(valid["combined"])
print("Validation:", evaluate(valid["label"], validation_pred)["macro_f1"])
test_pred = model.predict(test["combined"])
metrics = evaluate(test["label"], test_pred)
print(metrics["classification_report"])
print("Confusion matrix [fake, real]:", metrics["confusion_matrix"])

Validation: 1.0
              precision    recall  f1-score   support

        fake       1.00      1.00      1.00         4
        real       1.00      1.00      1.00         3

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7

Confusion matrix [fake, real]: [[4, 0], [0, 3]]


## 4 · Save this run

New outputs go under `results/sample/` or `results/full/`; supplied historical files stay intact.

In [10]:
scope = "sample" if DATA_MODE == "sample" else "full"
output_dir = save_results("tfidf_baseline", scope, test["label"], test_pred, model.predict_proba(test["combined"]))
print("Saved to:", output_dir)

Saved to: C:\Users\Mohd Shoeb\OneDrive\Desktop\Fake-News-Detection-System\results\sample\tfidf_baseline
